# Microkinetic Modeling of ORR on FePc


## Goal of the notebook

Let us connect an ORR free-energy landscape with surface coverage and current. We will also separate kinetic effects from oxygen-transport effects.


## Description of the method

We represent ORR on FePc with oxygen adsorption and four reversible electron-transfer steps. Intermediate free energies set reversible potentials. Rate expressions connect those potentials to reaction rates, and a steady-state site balance gives surface coverages.

The Levich equation estimates the oxygen-transport limit. The Koutecky–Levich relation combines that limit with kinetic current. Interactive controls update the free-energy diagram, disk current, and coverages together.


## What we will do

1. Let us define intermediate free energies and reversible rates, then solve the steady-state coverages.
2. We will calculate kinetic current and apply the rotating-disk transport correction.
3. We will use interactive plots to explore energy, rate, site density, and rotation effects.


In [ ]:
# Install course dependencies only in Google Colab. Local runs use environment.yml.
!if python -c "import google.colab" >/dev/null 2>&1; then pip install -q numpy scipy matplotlib ipywidgets; fi

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from functools import partial
from IPython.display import display
from ipywidgets import Checkbox, FloatSlider, HBox, interactive_output, VBox
from ipywidgets import Dropdown, fixed, FloatLogSlider, FloatSlider, interact
from scipy.linalg import solve

In [ ]:
# Standard publication figure sizes in inches
plt.rc("figure", figsize  = (8.25 / 2.54, 8.25 / 2.54))
plt.rc("figure", dpi      = 100)
plt.rc("lines",  linewidth= 4)
plt.rc("font",   size     = 10)
plt.rc("font",   family   = "sans-serif")
plt.rc("axes",   titlesize= 10)
plt.rc("axes",   labelsize= 9)
plt.rc("xtick",  labelsize= 8)
plt.rc("ytick",  labelsize= 8)
plt.rc("legend", fontsize = 8)
plt.rc("figure", titlesize= 10)

pd.options.display.float_format = "{:.2f}".format

In [ ]:
colors = ["#1965B0", "#7BAFDE", "#90C987", "#F1932D", "#DC050C"]

## Free-energy parameters

**Core exercise**

Let us use this simplified ORR path:

$$
\mathrm{\ast + O_2}
\rightarrow
\mathrm{OOH\ast}
\rightarrow
\mathrm{O\ast}
\rightarrow
\mathrm{OH\ast}
\rightarrow
\mathrm{\ast + H_2O}
$$

We set the final state to zero:

$$
G(\mathrm{\ast + H_2O}) = 0
$$


### Free-energy parameters (continued)

At zero potential, the initial state has energy

$$
G(\mathrm{\ast + O_2}) = 4.92\ \mathrm{eV}
$$

because

$$
4.92 = 4 \times 1.23
$$

We will vary two thermodynamic parameters:

$$
G_{\mathrm{OH}}, \qquad b_{\mathrm{OOH}}
$$


### Free-energy parameters (continued)

We also vary

$$
b_{\mathrm{O}}
$$

The scaling relations are

$$
G_{\mathrm{OOH}} = G_{\mathrm{OH}} + b_{\mathrm{OOH}}
$$

$$
G_{\mathrm{O}} = 2(G_{\mathrm{OH}} - b_{\mathrm{O}})
$$

At potential $U$, the first level is

$$
G_0 = 4.92 - 4U
$$


### Free-energy parameters (continued)

The remaining levels are

$$
G_1 = G_{\mathrm{OOH}} - 3U
$$

$$
G_2 = G_{\mathrm{O}} - 2U
$$

$$
G_3 = G_{\mathrm{OH}} - U
$$

$$
G_4 = 0
$$


In [ ]:
faraday_constant_c_per_mol = 96485.0
gas_constant_j_per_mol_k = 8.314
temperature_k = 295.15
thermal_voltage_v = (gas_constant_j_per_mol_k * temperature_k / faraday_constant_c_per_mol)
equilibrium_potential_v = 1.23
total_orr_free_energy_ev = 4.0 * equilibrium_potential_v
default_params = {
    "G_OH": 0.82,
    "b_OOH": 3.03,
    "b_O": -0.3,
    "k_ads": 2000.0,
    "K_ads": 0.6,
    "k_OOH": 3.0,
    "k_O": 0.01,
    "k_OH": 0.01,
    "k_free": 60.0,
    "Gamma": 1.2e-10,
}

In [ ]:
def intermediate_energies(G_OH, b_OOH, b_O):
    """
    Return adsorption energies:
        G_OOH, G_O, G_OH
    """
    G_OOH = G_OH + b_OOH
    G_O = 2.0 * (G_OH - b_O)
    return [G_OOH, G_O, G_OH]

In [ ]:
def free_energy_states(potential_v, intermediate_energies_ev):
    """
    Return the five free-energy levels:
        * + O2, OOH*, O*, OH*, * + H2O
    """
    return [
        total_orr_free_energy_ev - 4.0 * potential_v,
        intermediate_energies_ev[0] - 3.0 * potential_v,
        intermediate_energies_ev[1] - 2.0 * potential_v,
        intermediate_energies_ev[2] - 1.0 * potential_v,
        0.0,
    ]

In [ ]:
# enumerate gives both the item number (starting at zero) and the item itself.
def plot_at_potential(potential_v, intermediate_energies_ev, color="#0000cc", label="model"):
    """
    Plot one free-energy diagram at potential U.
    """
    states = free_energy_states(potential_v, intermediate_energies_ev)
    bar_x = [(-0.2, 0.2), (0.8, 1.2), (1.8, 2.2), (2.8, 3.2), (3.8, 4.2)]
    for i, (x0, x1) in enumerate(bar_x):
        plt.plot([x0, x1], [states[i], states[i]], c=color, linestyle="-", linewidth=4, label=f"{label}, U = {potential_v:.2f} V" if i == 0 else None)
    for i in range(4):
        plt.plot([bar_x[i][1], bar_x[i + 1][0]], [states[i], states[i + 1]], c=color, linestyle=":", linewidth=1)

In [ ]:
def plot_free_energy_diagram(G_OH, b_OOH, b_O, applied_potential_v=0.0):
    """
    Plot free-energy diagrams at U = 1.23 V, U = U_L, and a selected U.
    """
    intermediate_energies_ev = intermediate_energies(G_OH, b_OOH, b_O)
    step_energies_at_U0 = [
        total_orr_free_energy_ev - intermediate_energies_ev[0],
        intermediate_energies_ev[0] - intermediate_energies_ev[1],
        intermediate_energies_ev[1] - intermediate_energies_ev[2],
        intermediate_energies_ev[2],
    ]
    overpotential_v = equilibrium_potential_v - min(step_energies_at_U0)
    plt.figure()
    plot_at_potential(1.23, intermediate_energies_ev, colors[1], "Cat.")
    plot_at_potential(round(1.23 - overpotential_v, 2), intermediate_energies_ev, colors[2], "Cat.")
    plot_at_potential(applied_potential_v, intermediate_energies_ev, colors[0], "Cat.")
    plt.xlabel("Reaction coordinate")
    plt.ylabel("Free energies / eV")
    plt.xticks([0, 1, 2, 3, 4], ["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"])
    plt.ylim(-1, 5.5)
    plt.legend()
    plt.show()

In [ ]:
# Controls for plot_free_energy_diagram: changing a value calls the plot again.
plot_free_energy_diagram_controls = {}
plot_free_energy_diagram_controls['G_OH'] = FloatSlider(value=default_params['G_OH'], min=0.5, max=1.1, step=0.01, description='G_OH / eV', continuous_update=False)
plot_free_energy_diagram_controls['b_OOH'] = FloatSlider(value=default_params['b_OOH'], min=2.5, max=3.5, step=0.01, description='b_OOH / eV', continuous_update=False)

In [ ]:
plot_free_energy_diagram_controls['b_O'] = FloatSlider(value=default_params['b_O'], min=-0.8, max=0.2, step=0.01, description='b_O / eV', continuous_update=False)

### Change the binding energies

Move one energy slider and trace the corresponding intermediate in the diagram.


In [ ]:
plot_free_energy_diagram_controls["applied_potential_v"] = FloatSlider(value=0.0, min=-0.2, max=1.5, step=0.01, description="U / V vs RHE", continuous_update=False)

In [ ]:
# The workshop layout places controls beside the interactive result.
plot_free_energy_diagram_output = interactive_output(plot_free_energy_diagram, plot_free_energy_diagram_controls)
display(HBox([plot_free_energy_diagram_output, VBox(list(plot_free_energy_diagram_controls.values()))]))

## Steady-state coverages

**Core exercise**

The kinetic model uses the full surface cycle:

$$
1:\quad \mathrm{\ast + O_2}
\rightleftharpoons
\mathrm{O_2\ast}
$$

$$
2:\quad \mathrm{O_2\ast}
\rightleftharpoons
\mathrm{OOH\ast}
$$

$$
3:\quad \mathrm{OOH\ast}
\rightleftharpoons
\mathrm{O\ast}
$$


### Steady-state coverages (continued)

$$
4:\quad \mathrm{O\ast}
\rightleftharpoons
\mathrm{OH\ast}
$$

$$
5:\quad \mathrm{OH\ast}
\rightleftharpoons
\mathrm{\ast}
$$

The surface coverages are

$$
\theta_{\ast},
\theta_{\mathrm{O_2}},
\theta_{\mathrm{OOH}},
\theta_{\mathrm{O}},
\theta_{\mathrm{OH}}
$$


### Steady-state coverages (continued)

with the site balance

$$
\theta_{\ast}
+
\theta_{\mathrm{O_2}}
+
\theta_{\mathrm{OOH}}
+
\theta_{\mathrm{O}}
+
\theta_{\mathrm{OH}}
=
1
$$

The thermodynamic parameters are the same as in the free-energy diagram:

$$
G_{\mathrm{OOH}} = G_{\mathrm{OH}} + b_{\mathrm{OOH}}
$$


### Steady-state coverages (continued)

$$
G_{\mathrm{O}} = 2(G_{\mathrm{OH}} - b_{\mathrm{O}})
$$


The `rate_constants` code calculates reversible potentials from free-energy differences between neighboring states. The coverage solver repeats this calculation at every plotted potential.

> **Pause and predict:** Which intermediate could occupy most sites when OH binding is very strong?
>
> **What to look for:** At every potential, coverages should be nonnegative and sum to one. A departure points to a numerical or model-formulation issue.


In [ ]:
coverage_labels = [
    "$\\theta_{\\ast}$",
    "$\\theta_{\\mathrm{O_2}}$",
    "$\\theta_{\\mathrm{OOH}}$",
    "$\\theta_{\\mathrm{O}}$",
    "$\\theta_{\\mathrm{OH}}$",
]

In [ ]:
def electrochemical_rate(k0, potential_v, equilibrium_potentials_v, alpha=0.5):
    """
    Forward and reverse rate constants for one electron-transfer step.
    """
    k_forward = k0 * np.exp(-alpha * (potential_v - equilibrium_potentials_v) / thermal_voltage_v)
    k_reverse = k0 * np.exp((1.0 - alpha) * (potential_v - equilibrium_potentials_v) / thermal_voltage_v)
    return (k_forward, k_reverse)

In [ ]:
def rate_constants(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    """
    Rate constants for the natural ORR flow:
        1 adsorption
        2 OOH formation
        3 O formation
        4 OH formation
        5 free-site regeneration
    """
    # Free energy levels at zero potential, followed by adjacent step differences.
    G_OOH = G_OH + b_OOH
    G_O = 2.0 * (G_OH - b_O)
    adsorption_free_energy_ev = -thermal_voltage_v * np.log(K_ads)
    G_O2 = total_orr_free_energy_ev + adsorption_free_energy_ev
    equilibrium_potentials_v = {
        "E_OOH": G_O2 - G_OOH,
        "E_O": G_OOH - G_O,
        "E_OH": G_O - G_OH,
        "E_free": G_OH,
    }
    k1f = k_ads
    k1r = k_ads / K_ads
    k2f, k2r = electrochemical_rate(k_OOH, potential_v, equilibrium_potentials_v["E_OOH"])
    k3f, k3r = electrochemical_rate(k_O, potential_v, equilibrium_potentials_v["E_O"])
    k4f, k4r = electrochemical_rate(k_OH, potential_v, equilibrium_potentials_v["E_OH"])
    k5f, k5r = electrochemical_rate(k_free, potential_v, equilibrium_potentials_v["E_free"])
    return (k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r)

In [ ]:
def steady_state_coverages(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    """
    Solve coverages at one potential.

    Coverage order:
        *, O2*, OOH*, O*, OH*
    """
    k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r = rate_constants(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    rate_matrix = np.array(
        [
            [k1f, -(k1r + k2f), k2r, 0.0, 0.0],
            [0.0, k2f, -(k2r + k3f), k3r, 0.0],
            [0.0, 0.0, k3f, -(k3r + k4f), k4r],
            [k5r, 0.0, 0.0, k4f, -(k4r + k5f)],
            [1.0, 1.0, 1.0, 1.0, 1.0],
        ]
    )
    rhs = np.array([0.0, 0.0, 0.0, 0.0, 1.0])
    return solve(rate_matrix, rhs)

In [ ]:
def coverage_curves(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    coverages = np.zeros((len(potential_grid_v), 5))
    for i, potential_v in enumerate(potential_grid_v):
        coverages[i] = steady_state_coverages(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    return coverages

In [ ]:
# [:, column] selects that column for all rows of a NumPy array.
def plot_coverages(G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    potential_grid_v = np.linspace(1.0, 0.2, 500)
    coverages = coverage_curves(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    plt.figure()
    for i, label in enumerate(coverage_labels):
        plt.plot(potential_grid_v, coverages[:, i], label=label)
    plt.xlabel("U / V vs RHE")
    plt.ylabel("Coverage")
    plt.ylim(0, 1)
    plt.legend()
    plt.show()

In [ ]:
# Controls for plot_coverages: changing a value calls the plot again.
plot_coverages_controls = {}
plot_coverages_controls['G_OH'] = FloatSlider(value=default_params['G_OH'], min=0.5, max=1.1, step=0.01, description='G_OH', continuous_update=False)
plot_coverages_controls['b_OOH'] = FloatSlider(value=default_params['b_OOH'], min=2.5, max=3.5, step=0.01, description='b_OOH', continuous_update=False)

In [ ]:
plot_coverages_controls['b_O'] = FloatSlider(value=default_params['b_O'], min=-0.8, max=0.2, step=0.01, description='b_O', continuous_update=False)
plot_coverages_controls['k_ads'] = FloatLogSlider(value=default_params['k_ads'], base=10, min=1, max=5, step=0.1, description='k_ads', continuous_update=False)
plot_coverages_controls['K_ads'] = FloatSlider(value=default_params['K_ads'], min=0.05, max=3.0, step=0.05, description='K_ads', continuous_update=False)

In [ ]:
plot_coverages_controls['k_OOH'] = FloatLogSlider(value=default_params['k_OOH'], base=10, min=-2, max=3, step=0.1, description='k_OOH', continuous_update=False)
plot_coverages_controls['k_O'] = FloatLogSlider(value=default_params['k_O'], base=10, min=-3, max=3, step=0.1, description='k_O', continuous_update=False)

In [ ]:
plot_coverages_controls['k_OH'] = FloatLogSlider(value=default_params['k_OH'], base=10, min=-3, max=3, step=0.1, description='k_OH', continuous_update=False)
plot_coverages_controls['k_free'] = FloatLogSlider(value=default_params['k_free'], base=10, min=-2, max=4, step=0.1, description='k_free', continuous_update=False)

### Compare the surface populations

Which intermediate occupies most sites as the binding or rate parameters change?


In [ ]:
# The workshop layout places controls beside the interactive result.
plot_coverages_output = interactive_output(plot_coverages, plot_coverages_controls)
display(HBox([plot_coverages_output, VBox(list(plot_coverages_controls.values()))]))

## Kinetic current

**Core exercise**

The kinetic current is calculated from the electron-transfer steps in the surface cycle:

$$
1:\quad \mathrm{\ast + O_2}
\rightleftharpoons
\mathrm{O_2\ast}
$$

$$
2:\quad \mathrm{O_2\ast}
\rightleftharpoons
\mathrm{OOH\ast}
$$

$$
3:\quad \mathrm{OOH\ast}
\rightleftharpoons
\mathrm{O\ast}
$$


### Kinetic current (continued)

$$
4:\quad \mathrm{O\ast}
\rightleftharpoons
\mathrm{OH\ast}
$$

$$
5:\quad \mathrm{OH\ast}
\rightleftharpoons
\mathrm{\ast}
$$

Step 1 is chemical oxygen adsorption. Steps 2–5 each transfer one electron.

The kinetic current density is

$$
j_k =
-F\Gamma(r_2+r_3+r_4+r_5)
$$

Here $\Gamma$ is active-site concentration. The slider uses units of


### Kinetic current (continued)

$$
\mathrm{pmol\ cm^{-2}}
$$

so the code converts it as

$$
\Gamma_{\mathrm{mol}} =
\Gamma_{\mathrm{pmol}}\times 10^{-12}
$$


In [ ]:
def kinetic_current_curve(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma):
    current_values_a_cm2 = []
    for potential_v in potential_grid_v:
        # At each potential, find coverages before calculating the net electron flux.
        coverages = steady_state_coverages(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
        theta_free, theta_O2, theta_OOH, theta_O, theta_OH = coverages
        k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r = rate_constants(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
        # Step 1 is chemical adsorption; only steps 2 to 5 transfer electrons.
        reaction_rate_1 = k1f * theta_free - k1r * theta_O2
        reaction_rate_2 = k2f * theta_O2 - k2r * theta_OOH
        reaction_rate_3 = k3f * theta_OOH - k3r * theta_O
        reaction_rate_4 = k4f * theta_O - k4r * theta_OH
        reaction_rate_5 = k5f * theta_OH - k5r * theta_free
        electron_transfer_rates = np.array([reaction_rate_2, reaction_rate_3, reaction_rate_4, reaction_rate_5])
        current_a_cm2 = -faraday_constant_c_per_mol * gamma * np.sum(electron_transfer_rates)
        current_values_a_cm2.append(current_a_cm2)
    return np.array(current_values_a_cm2)

In [ ]:
def plot_kinetic_current(G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma):
    gamma_mol = gamma * 1e-12
    potential_grid_v = np.linspace(1.0, 0.2, 500)
    kinetic_current_a_cm2 = kinetic_current_curve(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol)
    plt.figure()
    plt.plot(potential_grid_v, 1000.0 * kinetic_current_a_cm2, color=colors[0])
    plt.xlabel("U / V vs RHE")
    plt.ylabel("$j_k$ / mA cm$^{-2}$")
    plt.ylim(-10, 0)
    plt.show()

In [ ]:
# Controls for plot_kinetic_current: changing a value calls the plot again.
plot_kinetic_current_controls = {}
plot_kinetic_current_controls['G_OH'] = FloatSlider(value=default_params['G_OH'], min=0.5, max=1.1, step=0.01, description='G_OH', continuous_update=False)
plot_kinetic_current_controls['b_OOH'] = FloatSlider(value=default_params['b_OOH'], min=2.5, max=3.5, step=0.01, description='b_OOH', continuous_update=False)

In [ ]:
plot_kinetic_current_controls['b_O'] = FloatSlider(value=default_params['b_O'], min=-0.8, max=0.2, step=0.01, description='b_O', continuous_update=False)
plot_kinetic_current_controls['k_ads'] = FloatLogSlider(value=default_params['k_ads'], base=10, min=1, max=5, step=0.1, description='k_ads', continuous_update=False)
plot_kinetic_current_controls['K_ads'] = FloatSlider(value=default_params['K_ads'], min=0.05, max=3.0, step=0.05, description='K_ads', continuous_update=False)

In [ ]:
plot_kinetic_current_controls['k_OOH'] = FloatLogSlider(value=default_params['k_OOH'], base=10, min=-2, max=3, step=0.1, description='k_OOH', continuous_update=False)
plot_kinetic_current_controls['k_O'] = FloatLogSlider(value=default_params['k_O'], base=10, min=-3, max=3, step=0.1, description='k_O', continuous_update=False)

In [ ]:
plot_kinetic_current_controls['k_OH'] = FloatLogSlider(value=default_params['k_OH'], base=10, min=-3, max=3, step=0.1, description='k_OH', continuous_update=False)
plot_kinetic_current_controls['k_free'] = FloatLogSlider(value=default_params['k_free'], base=10, min=-2, max=4, step=0.1, description='k_free', continuous_update=False)
plot_kinetic_current_controls['gamma'] = FloatSlider(value=default_params['Gamma'] / 1e-12, min=10.0, max=2000.0, step=10.0, description='gamma', continuous_update=False)

### Explore the kinetic current

Change one rate parameter at a time. Describe the current response before you move another control.


In [ ]:
# The workshop layout places controls beside the interactive result.
plot_kinetic_current_output = interactive_output(plot_kinetic_current, plot_kinetic_current_controls)
display(HBox([plot_kinetic_current_output, VBox(list(plot_kinetic_current_controls.values()))]))

## RDE transport correction

**Core exercise**

The kinetic model gives the current density controlled only by surface reaction rates:

$$
j_k
$$

In an RRDE experiment, the measured disk current is also limited by oxygen transport to the rotating disk.

The diffusion-limited current density is estimated with the Levich equation:

$$
j_L =
-0.62 n F D_{\mathrm{O_2}}^{2/3}
\nu^{-1/6}
C_{\mathrm{O_2}}
\omega^{1/2}
$$

where $n=4$ for four-electron ORR, $D_{\mathrm{O_2}}$ is the oxygen diffusion coefficient, $\nu$ is the kinematic viscosity, $C_{\mathrm{O_2}}$ is the bulk oxygen concentration, and $\omega$ is the rotation rate in rad s$^{-1}$:


### RDE transport correction (continued)

$$
\omega = \frac{2\pi\ \mathrm{rpm}}{60}
$$


The Koutecky–Levich relation combines kinetic and transport limits:

$$
\frac{1}{j_{\mathrm{disk}}}
=
\frac{1}{j_k}
+
\frac{1}{j_L}
$$

The plotted current is cathodic, so its value is negative.


In [ ]:
def levich_current_density(rpm, n=4, D_O2=2e-05, nu=0.01, C_O2=1.2e-06):
    """
    Levich diffusion-limited current density in A cm-2.
    """
    omega = 2.0 * np.pi * rpm / 60.0
    return (-0.62 * n * faraday_constant_c_per_mol * D_O2 ** (2.0 / 3.0) * nu ** (-1.0 / 6.0) * C_O2 * omega**0.5)

In [ ]:
def disk_current_density(kinetic_current_a_cm2, limiting_current_a_cm2):
    """
    Koutecky-Levich combination.

    Uses magnitudes internally and returns a cathodic current.
    """
    kinetic_current_magnitude = np.maximum(np.abs(kinetic_current_a_cm2), 1e-30)
    limiting_current_magnitude = max(abs(limiting_current_a_cm2), 1e-30)
    disk_current_magnitude = 1.0 / (1.0 / kinetic_current_magnitude + 1.0 / limiting_current_magnitude)
    return -disk_current_magnitude

In [ ]:
def plot_rrde_disk_current(G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma, rpm):
    gamma_mol = gamma * 1e-12
    potential_grid_v = np.linspace(1.0, 0.2, 500)
    kinetic_current_a_cm2 = kinetic_current_curve(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol)
    limiting_current_a_cm2 = levich_current_density(rpm)
    disk_current_a_cm2 = disk_current_density(kinetic_current_a_cm2, limiting_current_a_cm2)
    plt.figure()
    plt.plot(potential_grid_v, 1000.0 * disk_current_a_cm2, color=colors[0])
    plt.axhline(1000.0 * limiting_current_a_cm2, color=colors[4], linestyle=":", linewidth=1)
    plt.xlabel("U / V vs RHE")
    plt.ylabel("$j_{\\mathrm{disk}}$ / mA cm$^{-2}$")
    plt.ylim(-10, 0)
    plt.show()
    print(f"rpm = {rpm}")
    print(f"sqrt(rpm) = {np.sqrt(rpm):.2f}")
    print(f"j_L = {1000.0 * limiting_current_a_cm2:.2f} mA cm-2")
    print(f"most cathodic j_k = {1000.0 * np.min(kinetic_current_a_cm2):.2f} mA cm-2")

In [ ]:
# Controls for plot_rrde_disk_current: changing a value calls the plot again.
plot_rrde_disk_current_controls = {}
plot_rrde_disk_current_controls['G_OH'] = FloatSlider(value=default_params['G_OH'], min=0.5, max=1.1, step=0.01, description='G_OH', continuous_update=False)
plot_rrde_disk_current_controls['b_OOH'] = FloatSlider(value=default_params['b_OOH'], min=2.5, max=3.5, step=0.01, description='b_OOH', continuous_update=False)

In [ ]:
plot_rrde_disk_current_controls['b_O'] = FloatSlider(value=default_params['b_O'], min=-0.8, max=0.2, step=0.01, description='b_O', continuous_update=False)
plot_rrde_disk_current_controls['k_ads'] = FloatLogSlider(value=default_params['k_ads'], base=10, min=1, max=5, step=0.1, description='k_ads', continuous_update=False)
plot_rrde_disk_current_controls['K_ads'] = FloatSlider(value=default_params['K_ads'], min=0.05, max=3.0, step=0.05, description='K_ads', continuous_update=False)

In [ ]:
plot_rrde_disk_current_controls['k_OOH'] = FloatLogSlider(value=default_params['k_OOH'], base=10, min=-2, max=3, step=0.1, description='k_OOH', continuous_update=False)
plot_rrde_disk_current_controls['k_O'] = FloatLogSlider(value=default_params['k_O'], base=10, min=-3, max=3, step=0.1, description='k_O', continuous_update=False)

In [ ]:
plot_rrde_disk_current_controls['k_OH'] = FloatLogSlider(value=default_params['k_OH'], base=10, min=-3, max=3, step=0.1, description='k_OH', continuous_update=False)
plot_rrde_disk_current_controls['k_free'] = FloatLogSlider(value=default_params['k_free'], base=10, min=-2, max=4, step=0.1, description='k_free', continuous_update=False)
plot_rrde_disk_current_controls['gamma'] = FloatSlider(value=default_params['Gamma'] / 1e-12, min=10.0, max=2000.0, step=10.0, description='gamma', continuous_update=False)

In [ ]:
plot_rrde_disk_current_controls['rpm'] = Dropdown(options=[400, 900, 1600, 2500, 3600], value=1600, description='rpm')

### Change the rotation rate

Hold the surface parameters fixed and vary rotation. At what point does transport set the disk current?


In [ ]:
# The workshop layout places controls beside the interactive result.
plot_rrde_disk_current_output = interactive_output(plot_rrde_disk_current, plot_rrde_disk_current_controls)
display(HBox([plot_rrde_disk_current_output, VBox(list(plot_rrde_disk_current_controls.values()))]))

## Interactive summary dashboard

**Core exercise**

Let us bring the three outputs together. The sliders call `plot_summary_dashboard` again after each change, so the function recalculates and redraws each panel.

The same parameters update:

1. the free-energy diagram,
2. the RDE disk current,
3. the steady-state coverages.

The thermodynamic controls set the relative energies:

$$
G_{\mathrm{OH}},\quad b_{\mathrm{OOH}},\quad b_{\mathrm{O}}
$$

The kinetic controls set the surface reaction rates:

$$
k_{\mathrm{ads}},\quad k_{\mathrm{OOH}},\quad k_{\mathrm{O}},\quad k_{\mathrm{OH}},\quad k_{\mathrm{free}}
$$


### Interactive summary dashboard (continued)

The RDE parameters are the active-site concentration

$$
\Gamma
$$


in pmol cm$^{-2}$, and the rotation rate in rpm. The disk current follows

$$
\frac{1}{j_{\mathrm{disk}}}
=
\frac{1}{j_k}
+
\frac{1}{j_L}
$$

> **Explore:** Change one slider family at a time: thermodynamics, kinetics, site density, then rotation rate.
>
> **What to look for:** Explain which output changes in position, shape, or scale, and connect each change to its parameter.


In [ ]:
# zip pairs items from the supplied lists in the same order.
def plot_summary_dashboard(G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma, rpm, applied_potential_v=0.0):
    gamma_mol = gamma * 1e-12
    potential_grid_v = np.linspace(1.0, 0.2, 500)
    coverages = coverage_curves(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    kinetic_current_a_cm2 = kinetic_current_curve(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol)
    limiting_current_a_cm2 = levich_current_density(rpm)
    disk_current_a_cm2 = disk_current_density(kinetic_current_a_cm2, limiting_current_a_cm2)
    fig, axes = plt.subplots(1, 3, figsize=(25.0 / 2.54, 8.25 / 2.54))
    # Draw the energy panel directly beside the current and coverage calculations.
    ax = axes[0]
    intermediate_energies_ev = intermediate_energies(G_OH, b_OOH, b_O)
    step_energies_at_U0 = [
        total_orr_free_energy_ev - intermediate_energies_ev[0],
        intermediate_energies_ev[0] - intermediate_energies_ev[1],
        intermediate_energies_ev[1] - intermediate_energies_ev[2],
        intermediate_energies_ev[2],
    ]
    overpotential_v = equilibrium_potential_v - min(step_energies_at_U0)
    U_values = [1.23, round(1.23 - overpotential_v, 2), applied_potential_v]
    U_colors = [colors[1], colors[2], colors[0]]
    bar_x = [(-0.2, 0.2), (0.8, 1.2), (1.8, 2.2), (2.8, 3.2), (3.8, 4.2)]
    for potential_v, color in zip(U_values, U_colors):
        states = free_energy_states(potential_v, intermediate_energies_ev)
        for i, (x0, x1) in enumerate(bar_x):
            ax.plot([x0, x1], [states[i], states[i]], c=color, linestyle="-", linewidth=4, label=f"U = {potential_v:.2f} V" if i == 0 else None)
        for i in range(4):
            ax.plot([bar_x[i][1], bar_x[i + 1][0]], [states[i], states[i + 1]], c=color, linestyle=":", linewidth=1)
    ax.set_xlabel("Reaction coordinate")
    ax.set_ylabel("Free energies / eV")
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.set_xticklabels(["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"])
    ax.set_ylim(-1, 5.5)
    ax.legend()
    ax.set_title("Free-energy diagram")
    disk_current_ma_cm2 = 1000.0 * disk_current_a_cm2
    # Interpolate the first crossing of the midpoint current.
    x = potential_grid_v
    y = disk_current_ma_cm2
    y_start = y[0]
    y_end = y[-1]
    y_half = 0.5 * (y_start + y_end)
    x_half = None
    for i in range(len(x) - 1):
        y0 = y[i]
        y1 = y[i + 1]
        if (y0 - y_half) * (y1 - y_half) <= 0:
            if y1 == y0:
                x_half = 0.5 * (x[i] + x[i + 1])
            else:
                x_half = x[i] + (y_half - y0) * (x[i + 1] - x[i]) / (y1 - y0)
            break
    half_wave_potential_v = x_half
    half_wave_current_ma_cm2 = y_half
    axes[1].plot(potential_grid_v, disk_current_ma_cm2, color=colors[0])
    axes[1].axhline(1000.0 * limiting_current_a_cm2, color=colors[4], linestyle=":", linewidth=1)
    if half_wave_potential_v is not None:
        axes[1].axhline(half_wave_current_ma_cm2, color=colors[3], linestyle="--", linewidth=1)
        axes[1].axvline(half_wave_potential_v, color=colors[3], linestyle="--", linewidth=1)
        axes[1].plot(half_wave_potential_v, half_wave_current_ma_cm2, marker="o", color=colors[3])
        axes[1].text(half_wave_potential_v, half_wave_current_ma_cm2, f"  E$_{{1/2}}$ = {half_wave_potential_v:.2f} V", va="bottom", ha="left")
    axes[1].set_xlabel("U / V vs RHE")
    axes[1].set_ylabel("$j_{\\mathrm{disk}}$ / mA cm$^{-2}$")
    axes[1].set_ylim(-10, 0)
    axes[1].set_title(f"RRDE disk current, {rpm} rpm")
    for i, label in enumerate(coverage_labels):
        axes[2].plot(potential_grid_v, coverages[:, i], label=label)
    axes[2].set_xlabel("U / V vs RHE")
    axes[2].set_ylabel("Coverage")
    axes[2].set_ylim(0, 1)
    axes[2].legend()
    axes[2].set_title("Surface coverages")
    plt.tight_layout()
    plt.show()

In [ ]:
# Controls for plot_summary_dashboard: changing a value calls the plot again.
plot_summary_dashboard_controls = {}
plot_summary_dashboard_controls['G_OH'] = FloatSlider(value=default_params['G_OH'], min=0.5, max=1.1, step=0.01, description='G_OH', continuous_update=False)
plot_summary_dashboard_controls['b_OOH'] = FloatSlider(value=default_params['b_OOH'], min=2.5, max=3.5, step=0.01, description='b_OOH', continuous_update=False)

In [ ]:
plot_summary_dashboard_controls['b_O'] = FloatSlider(value=default_params['b_O'], min=-0.8, max=0.2, step=0.01, description='b_O', continuous_update=False)
plot_summary_dashboard_controls['k_ads'] = FloatLogSlider(value=default_params['k_ads'], base=10, min=1, max=5, step=0.1, description='k_ads', continuous_update=False)
plot_summary_dashboard_controls['K_ads'] = FloatSlider(value=default_params['K_ads'], min=0.05, max=3.0, step=0.05, description='K_ads', continuous_update=False)

In [ ]:
plot_summary_dashboard_controls['k_OOH'] = FloatLogSlider(value=default_params['k_OOH'], base=10, min=-2, max=3, step=0.1, description='k_OOH', continuous_update=False)
plot_summary_dashboard_controls['k_O'] = FloatLogSlider(value=default_params['k_O'], base=10, min=-3, max=3, step=0.1, description='k_O', continuous_update=False)

In [ ]:
plot_summary_dashboard_controls['k_OH'] = FloatLogSlider(value=default_params['k_OH'], base=10, min=-3, max=3, step=0.1, description='k_OH', continuous_update=False)
plot_summary_dashboard_controls['k_free'] = FloatLogSlider(value=default_params['k_free'], base=10, min=-2, max=4, step=0.1, description='k_free', continuous_update=False)
plot_summary_dashboard_controls['gamma'] = FloatSlider(value=default_params['Gamma'] / 1e-12, min=10.0, max=2000.0, step=10.0, description='gamma', continuous_update=False)

In [ ]:
plot_summary_dashboard_controls['rpm'] = Dropdown(options=[400, 900, 1600, 2500, 3600], value=1600, description='rpm')

### Compare all three panels

Use one parameter change to explain the free-energy diagram, disk current, and surface coverage together.


In [ ]:
plot_summary_dashboard_controls["applied_potential_v"] = FloatSlider(value=0.0, min=-0.2, max=1.5, step=0.01, description="U / V vs RHE", continuous_update=False)

In [ ]:
# The workshop layout places controls beside the interactive result.
plot_summary_dashboard_output = interactive_output(plot_summary_dashboard, plot_summary_dashboard_controls)
display(HBox([plot_summary_dashboard_output, VBox(list(plot_summary_dashboard_controls.values()))]))

### Check the site balance

For each potential, add the surface fractions. Their sum should equal one, and each fraction should be zero or positive.


In [ ]:
# Inspect the same default model used by the sliders at three potentials.
p = default_params
potential_check = np.array([0.9, 0.7, 0.5])
coverage_check = coverage_curves(potential_check, p["G_OH"], p["b_OOH"], p["b_O"], p["k_ads"], p["K_ads"], p["k_OOH"], p["k_O"], p["k_OH"], p["k_free"])
coverage_table = pd.DataFrame(coverage_check, columns=["free", "O2", "OOH", "O", "OH"])
coverage_table["potential_V"] = potential_check
coverage_table["coverage_sum"] = coverage_check.sum(axis=1)
display(coverage_table)
print("Coverage sums should equal one; no coverage should be negative.")

## Student exercise: Distinguish kinetics from transport

**Core exercise**

Record a baseline current curve and its coverages. Change rotation alone; then restore it and change one kinetic parameter. Compare disk current, coverages, and free-energy diagram. Explain which response follows oxygen transport and which follows surface kinetics. Use the existing controls; the solver needs no changes.

Record one prediction, one numerical or graphical result, and your chemical interpretation. Use copies of existing values so the main calculation stays intact.


In [ ]:
# Use the existing sliders above; additional code is optional.
# Record the baseline and changed settings in your written answer below.

### Your answer

Prediction: [write your prediction]

Result: [report a value or refer to your figure]

Chemical explanation: [explain the result]

Assumption or limitation: [state one limitation]


## What have you learned

Let us gather the main ideas from this notebook. You can now:

- Connect intermediate free energies to reversible potentials and elementary rate expressions.
- Check that surface fractions are nonnegative and satisfy the site balance.
- Explain the distinction between kinetic current and modeled disk current.
- Use parameter changes to explore thermodynamic, kinetic, site-density, and transport effects.


### Questions for discussion

1. Which panels change when rotation rate rises while all surface parameters stay fixed?
2. How could coverage, site-density, or rotation measurements help distinguish parameter sets with similar current curves?
